# 🧵 スパースモデリングとグループラッソ入門

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Murakami-Shoya/study-notebooks/blob/main/notebooks/2026-10-20_group-lasso.ipynb)

**出典**: [Notion教材ページ](https://app.notion.com/p/3f43834a206d81d98a78c761f713daea)（理論・数式・用語解説はNotionを参照）

**日付**: 2026-10-20

In [ ]:
# 環境検出（ローカル or Colab）
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    pass
print(f"Environment: {'Google Colab' if IN_COLAB else 'Local'}")

## 何を確かめるか

1. Lassoは係数をちょうど0にするが，Ridgeは0にしない
2. Group LassoはISTA（近接勾配法）で解け，グループ単位で係数が0になる
3. 交差検証でλを選ぶと検証誤差が最小になる

## どんなデータを用意するか

**データセット1（Lasso vs Ridge用）**: 10特徴のうち3つ（特徴0, 2, 5）だけが効く線形回帰データ

| 変数 | 形（shape） | 中身 |
|------|-------------|------|
| `X` | `(n_trials, n_features)` = `(100, 10)` | trials x features |
| `y` | `(n_trials,)` = `(100,)` | target variable |
| `w_true` | `(n_features,)` = `(10,)` | true coefficients (3 nonzero) |

**データセット2（Group Lasso用）**: 10電極 × 4帯域 = 40特徴，電極0と1だけが効く

| 変数 | 形（shape） | 中身 |
|------|-------------|------|
| `X` | `(n_trials, n_features)` = `(200, 40)` | trials x features (10 electrodes x 4 bands) |
| `y` | `(n_trials,)` = `(200,)` | target variable |
| `w_true` | `(n_features,)` = `(40,)` | true coefficients (8 nonzero: electrode 0,1 only) |
| `groups` | `(n_features,)` = `(40,)` | group ID per feature (0-9 for electrodes) |

## シナリオ一覧

| # | 使うデータ | 何を動かすか | 確かめること | 期待する結果 | 解説の節 |
|---|-----------|-------------|-------------|-------------|----------|
| S1 | データセット1 | Lasso vs Ridgeの正則化経路 | Lassoは0，Ridgeは0にならない | Lasso係数は途中で0，Ridge係数は0に近づくが0にならない | [§2](https://app.notion.com/p/3f43834a206d81d98a78c761f713daea#142-1) |
| S2 | データセット2 | Group Lasso ISTA実装 | グループ単位で0になる | 電極0,1だけ残り，他8電極は全係数0 | [§3](https://app.notion.com/p/3f43834a206d81d98a78c761f713daea#142-2), [§5](https://app.notion.com/p/3f43834a206d81d98a78c761f713daea#142-4) |
| S3 | データセット2 | 交差検証でλ選択 | CV MSE最小のλが見つかる | λ候補の中で検証誤差最小点が存在 | [§6](https://app.notion.com/p/3f43834a206d81d98a78c761f713daea#142-5) |

## 準備：データ生成とヘルパー関数

In [ ]:
# ライブラリのインポート
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

In [ ]:
# データセット1生成: (n_trials, n_features) -> (n_trials,), (n_features,)
def make_simple_data(n_trials=100, n_features=10, active_idx=[0, 2, 5], seed=42):
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((n_trials, n_features))
    w_true = np.zeros(n_features)
    w_true[active_idx] = [2.0, -1.5, 1.0]
    y = X @ w_true + 0.5 * rng.standard_normal(n_trials)  # (100,10) @ (10,) -> (100,)
    print(f"X: {X.shape}, y: {y.shape}, w_true: {w_true.shape}")
    print(f"Nonzero coefficients in w_true: {np.sum(w_true != 0)}/{n_features}")
    return X, y, w_true

In [ ]:
# データセット2生成: (n_trials, n_features) -> (n_trials,), (n_features,), (n_features,)
def make_grouped_data(n_trials=200, n_electrodes=10, n_bands=4, active_elec=[0, 1], seed=0):
    rng = np.random.default_rng(seed)
    n_features = n_electrodes * n_bands
    X = rng.standard_normal((n_trials, n_features))
    w_true = np.zeros(n_features)
    # 効く電極のみ係数を設定
    for e in active_elec:
        w_true[e * n_bands:(e + 1) * n_bands] = rng.uniform(-1, 1, n_bands)
    y = X @ w_true + 0.5 * rng.standard_normal(n_trials)  # (200,40) @ (40,) -> (200,)
    groups = np.repeat(np.arange(n_electrodes), n_bands)
    print(f"X: {X.shape}, y: {y.shape}, w_true: {w_true.shape}, groups: {groups.shape}")
    return X, y, w_true, groups

In [ ]:
# Group Lasso ISTA: (n_trials, n_features), (n_trials,), (n_features,) -> (n_features,)
def group_lasso_ista(X, y, groups, lam, n_iter=500, tol=1e-6):
    n_trials, n_features = X.shape
    w, L, eta = np.zeros(n_features), np.linalg.norm(X, ord=2)**2/n_trials, None
    eta = 1.0 / L
    for _ in range(n_iter):
        w_old, z = w.copy(), w - eta * X.T @ (X @ w - y) / n_trials  # 勾配ステップ
        # ブロック軟閾値
        for g in np.unique(groups):
            idx, norm_z = groups == g, np.linalg.norm(z[groups == g])
            thr = lam * np.sqrt(idx.sum()) * eta
            w[idx] = (1 - thr / norm_z) * z[idx] if norm_z > thr else 0.0
        if np.linalg.norm(w - w_old) < tol:
            break
    return w

In [ ]:
# 非ゼロ係数を持つグループIDのリストを返す
def active_groups(w, groups, threshold=1e-8):
    return [g for g in np.unique(groups) if np.any(np.abs(w[groups == g]) > threshold)]

## S1. Lasso vs Ridge正則化経路

Lassoは係数をちょうど0にし，Ridgeは0に近づけるが0にはしない（[Notion §2](https://app.notion.com/p/3f43834a206d81d98a78c761f713daea#142-1)）．

In [ ]:
# データ準備と標準化
X, y, _ = make_simple_data()
X_scaled = StandardScaler().fit_transform(X)
print("X_scaled:", X_scaled.shape)

In [ ]:
# λを変えながらLassoとRidgeを学習し，係数経路を記録
n_lambdas = 50
alphas = np.logspace(-3, 1, n_lambdas)
lasso_coefs = np.array([Lasso(alpha=a, max_iter=10000).fit(X_scaled, y).coef_ for a in alphas])
ridge_coefs = np.array([Ridge(alpha=a).fit(X_scaled, y).coef_ for a in alphas])
print("lasso_coefs:", lasso_coefs.shape, "ridge_coefs:", ridge_coefs.shape)

In [ ]:
# 係数経路をプロット
n_features = X.shape[1]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for i in range(n_features):
    axes[0].plot(alphas, lasso_coefs[:, i], alpha=0.7, linewidth=1)
    axes[1].plot(alphas, ridge_coefs[:, i], alpha=0.7, linewidth=1)
for ax in axes:
    ax.set_xscale('log'); ax.axhline(0, color='k', linestyle='--', linewidth=0.5)
    ax.set_xlabel('lambda'); ax.set_ylabel('Coefficient'); ax.grid(True, alpha=0.3)
axes[0].set_title('Lasso')
axes[1].set_title('Ridge')
plt.tight_layout()
plt.show()

**観察**: Lasso係数は途中で0，Ridge係数は0に近づくが0にならない．

In [ ]:
# α=0.1で学習し，非ゼロ係数数とshapeを確認
lasso_w = Lasso(alpha=0.1, max_iter=10000).fit(X_scaled, y).coef_
ridge_w = Ridge(alpha=0.1).fit(X_scaled, y).coef_
print(f"lasso_w: {lasso_w.shape}, ridge_w: {ridge_w.shape}")
print(f"Lasso nonzero: {np.sum(np.abs(lasso_w) > 1e-5)}/{n_features}")
print(f"Ridge nonzero: {np.sum(np.abs(ridge_w) > 1e-5)}/{n_features}")
# Lassoは一部が0，Ridgeは全て非ゼロ
assert np.sum(np.abs(lasso_w) < 1e-5) > 0
assert np.all(np.abs(ridge_w) > 1e-5)
assert lasso_w.shape == (n_features,)
assert ridge_w.shape == (n_features,)

## S2. Group Lasso実装（ISTA）

ISTA近接勾配法でGroup Lassoを解く（[Notion §5](https://app.notion.com/p/3f43834a206d81d98a78c761f713daea#142-4)）．グループごと0になることを確認．

In [ ]:
# データセット2を生成し標準化
X_g, y_g, w_true_g, groups = make_grouped_data()
n_electrodes = 10
n_bands = 4
n_features_g = n_electrodes * n_bands
X_g_scaled = StandardScaler().fit_transform(X_g)
print("X_g_scaled:", X_g_scaled.shape)

In [ ]:
# Group Lasso ISTAで係数を推定
w_glasso = group_lasso_ista(X_g_scaled, y_g, groups, lam=0.1)
print("w_glasso:", w_glasso.shape)
active_elec = active_groups(w_glasso, groups)
print(f"Active electrodes: {active_elec}")
print(f"Nonzero coefficients: {np.sum(np.abs(w_glasso) > 1e-8)}/{n_features_g}")

**観察**: 電極0, 1が選ばれ，他8電極は全係数0（グループ単位で選択）．

In [ ]:
# グループ選択の検証
assert set(active_elec) == {0, 1}, "Should select electrodes 0 and 1"
# 選ばれなかった電極は全係数0
for e in range(2, n_electrodes):
    assert np.all(np.abs(w_glasso[groups == e]) < 1e-8), f"Electrode {e} should be zero"
assert w_glasso.shape == (n_features_g,)

## S3. 交差検証でλ選択

5-fold CVで検証誤差が最小になるλを見つける（[Notion §6](https://app.notion.com/p/3f43834a206d81d98a78c761f713daea#142-5)）．

In [ ]:
# λの候補を対数スケールで生成
lambda_max = max([np.linalg.norm(X_g_scaled[:, groups == g].T @ y_g) / 
                  (len(y_g) * np.sqrt(n_bands)) for g in range(n_electrodes)])
n_lambdas_cv = 20
lambdas = np.logspace(np.log10(lambda_max), -3, n_lambdas_cv)
print(f"lambdas: {lambdas.shape}")

In [ ]:
# 5-fold CVで各λの検証誤差を計算
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_mse = []
for lam in lambdas:
    errors = []
    for train_idx, val_idx in kf.split(X_g_scaled):
        Xtr, Xv = X_g_scaled[train_idx], X_g_scaled[val_idx]  # (n_train,40), (n_val,40)
        ytr, yv = y_g[train_idx], y_g[val_idx]  # (n_train,), (n_val,)
        w = group_lasso_ista(Xtr, ytr, groups, lam=lam)  # (40,)
        pred = Xv @ w  # (n_val,40) @ (40,) -> (n_val,)
        errors.append(mean_squared_error(yv, pred))
    cv_mse.append(np.mean(errors))
cv_mse = np.array(cv_mse)  # (n_lambdas_cv,)
print("cv_mse:", cv_mse.shape)

In [ ]:
# 最小CV MSEのλを特定
best_idx = np.argmin(cv_mse)
print(f"Best lambda: {lambdas[best_idx]:.4f}, CV MSE: {cv_mse[best_idx]:.4f}")

In [ ]:
# CV曲線をプロット
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(lambdas, cv_mse, 'o-', alpha=0.7)
ax.axvline(lambdas[best_idx], color='r', linestyle='--', alpha=0.5, label='Best lambda')
ax.set_xscale('log')
ax.set_xlabel('lambda (log scale)')
ax.set_ylabel('CV Mean Squared Error')
ax.set_title('Cross-validation')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**観察**: CV MSEが最小になるλが存在し，赤破線で示される．

In [ ]:
# 最適λで誤差が改善されることを確認
assert cv_mse[best_idx] < cv_mse[0], "Best lambda should reduce CV error"
assert cv_mse.shape == (n_lambdas_cv,)

## まとめ

Lassoは係数を0にし，Group LassoはグループごとISTAで解ける．CVでλを選び，ECoGの電極・帯域選択に応用可能．